# Context Injection for arXiv Query Generation — Full Analysis

This notebook runs the complete benchmark and tells the story in one place:
1. **Team expansion** — can context map fictional team names to real author queries? *(primary metric)*
2. **Date resolution** — does a project phase name become the correct `submittedDate` range?
3. **Category correctness** — do category hints steer models to the right arXiv codes?
4. **The structural paradox** — why baseline *looks* better on validity but is semantically empty
5. **Complexity cost** — does combining multiple context types break smaller models?
6. **Model size** — do larger models follow contextual instructions more reliably?

## Experimental Design

All domain terms are **fictional** — LLMs cannot know them from training data:

| Context type | Fictional entries | What it tests |
|---|---|---|
| **Glossary** | SPARC, CryoNet, TurboPEC, FoldX, APEX | Acronym disambiguation |
| **Teams** | Helios, QuantumBio | Team name → author list expansion |
| **Project Phases** | Helios Phase 1/2, QuantumBio Pilot | Named period → `submittedDate` range |
| **Instruments** | NEXUS-II, PolariScope | Facility → arXiv category hints |
| **Datasets** | PHOTON-Z, SynBio-DB | Survey → arXiv category hints |

Each model runs every query in two modes: **baseline** (no context) and **context-enriched**.
Because the terms are fictional, any improvement in context mode is directly attributable to the injection mechanism.

In [ ]:
import sys
import os
import asyncio
import time
import re
import json
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.application.usecases.arxiv import run_arxiv_agent
from src.query_validator import validate_query_components
from src.domain.output_cleaner import clean_components
from src.domain.context.context_resolver import ContextResolver

## 1. Configuration

In [ ]:
CONTEXT_YAML_PATH = os.path.join('..', 'src', 'domain', 'context', 'context.yaml')
OUTPUT_CSV        = 'article_benchmark_results.csv'
OVERWRITE_CSV     = True

## 2. Context Preview

Verify the resolver loads all entries correctly.

In [ ]:
resolver = ContextResolver(CONTEXT_YAML_PATH)
print(f'Loaded {resolver.num_entries} context entries')
print()
for section, terms in resolver.get_all_terms().items():
    print(f'  {section:18s}: {", ".join(terms)}')

## 3. Test Queries

Organised by context type, then complexity. Each query is tagged with:
- `context_types`: which context sections it exercises
- `complexity`: `single` (one type), `multi` (two or more), or `control` (real terms, no injection needed)

In [ ]:
test_queries = [
    # ── GLOSSARY ──────────────────────────────────────────────────────────────
    {'query': 'Papers about SPARC analysis methods for galaxy surveys',
     'context_types': ['glossary'], 'complexity': 'single',
     'note': 'SPARC = Stellar Population Analysis. Should use astro-ph.GA.'},
    {'query': 'Papers about CryoNet architecture for molecular reconstruction',
     'context_types': ['glossary'], 'complexity': 'single',
     'note': 'CryoNet = cryo-EM neural network. Should map to q-bio.BM + cs.LG.'},
    {'query': 'Papers about TurboPEC simulations of emission lines in galaxy clusters',
     'context_types': ['glossary'], 'complexity': 'single',
     'note': 'TurboPEC = Turbulent Photoionization Emission Code.'},
    {'query': 'Papers about FoldX predictions for enzyme stability',
     'context_types': ['glossary'], 'complexity': 'single',
     'note': 'FoldX = quantum protein stability toolkit. Should suggest quant-ph + q-bio.BM.'},
    {'query': 'Papers about APEX transit spectroscopy of hot Jupiters',
     'context_types': ['glossary'], 'complexity': 'single',
     'note': 'APEX = Adaptive Photometric Exoplanet Surveyor.'},
    # ── TEAM ──────────────────────────────────────────────────────────────────
    {'query': 'Papers published by the Helios team about dark matter in galaxy clusters',
     'context_types': ['team'], 'complexity': 'single',
     'note': 'Helios → Vasquez-Torres, Nakamura, Sharma-Patel, OBrien-Walsh'},
    {'query': 'Papers authored by QuantumBio members on quantum circuit design',
     'context_types': ['team'], 'complexity': 'single',
     'note': 'QuantumBio → Chen-Rodriguez, Okafor, Petrov'},
    # ── PROJECT PHASE ─────────────────────────────────────────────────────────
    {'query': 'Papers published during Phase 1 of the Helios Project',
     'context_types': ['project_phase'], 'complexity': 'single',
     'note': 'Date range: 2020-06-01 to 2022-03-31'},
    {'query': 'Research during Phase 2 of the Helios Project about exoplanet characterization',
     'context_types': ['project_phase'], 'complexity': 'single',
     'note': 'Date range: 2022-04-01 to 2024-09-30'},
    {'query': 'Papers from the QuantumBio Pilot phase on enzyme kinetics',
     'context_types': ['project_phase'], 'complexity': 'single',
     'note': 'Date range: 2023-01-01 to 2023-12-31'},
    # ── INSTRUMENT ────────────────────────────────────────────────────────────
    {'query': 'Papers analyzing high-energy sources using NEXUS-II detector',
     'context_types': ['instrument'], 'complexity': 'single',
     'note': 'NEXUS-II → astro-ph.HE, astro-ph.IM'},
    {'query': 'Papers using PolariScope for magnetic field observations',
     'context_types': ['instrument'], 'complexity': 'single',
     'note': 'PolariScope → astro-ph.GA, astro-ph.SR, astro-ph.IM'},
    # ── DATASET ───────────────────────────────────────────────────────────────
    {'query': 'Papers analyzing stellar populations from the PHOTON-Z survey',
     'context_types': ['dataset'], 'complexity': 'single',
     'note': 'PHOTON-Z → astro-ph.GA, astro-ph.SR'},
    {'query': 'Papers using data from the SynBio-DB biological database',
     'context_types': ['dataset'], 'complexity': 'single',
     'note': 'SynBio-DB → q-bio, cs.LG'},
    # ── MULTI-CONTEXT ─────────────────────────────────────────────────────────
    {'query': 'Papers by the Helios team about SPARC analysis using NEXUS-II data',
     'context_types': ['team', 'glossary', 'instrument'], 'complexity': 'multi',
     'note': 'Three context types: team + glossary + instrument'},
    {'query': 'Papers published by QuantumBio during the Pilot phase using SynBio-DB',
     'context_types': ['team', 'project_phase', 'dataset'], 'complexity': 'multi',
     'note': 'Three context types: team + date range + dataset'},
    {'query': 'Dark matter research by the Helios team during Phase 1',
     'context_types': ['team', 'project_phase'], 'complexity': 'multi',
     'note': 'Two context types: team + date range'},
    # ── CONTROL ───────────────────────────────────────────────────────────────
    {'query': 'Papers about machine learning and neural networks',
     'context_types': ['control'], 'complexity': 'control',
     'note': 'Real terms — baseline and context should be equivalent'},
    {'query': 'Papers on quantum computing and quantum algorithms',
     'context_types': ['control'], 'complexity': 'control',
     'note': 'Real terms — baseline and context should be equivalent'},
    {'query': 'Papers about protein structure prediction',
     'context_types': ['control'], 'complexity': 'control',
     'note': 'Real terms — baseline and context should be equivalent'},
    {'query': 'Papers authored by John Smith on gravitational waves',
     'context_types': ['control_author'], 'complexity': 'control',
     'note': 'Generic author, no team — should produce au:smith'},
]

print(f'{len(test_queries)} queries: '
      f'{sum(1 for q in test_queries if q["complexity"]=="single")} single, '
      f'{sum(1 for q in test_queries if q["complexity"]=="multi")} multi, '
      f'{sum(1 for q in test_queries if q["complexity"]=="control")} control')

In [ ]:
# Verify the resolver picks up the right context entries for each query
for tq in test_queries:
    resolved = resolver.resolve(tq['query'])
    matches = []
    if resolved.glossary:    matches.extend([f'glossary:{g.term}' for g in resolved.glossary])
    if resolved.teams:       matches.extend([f'team:{t.code}({len(t.members)}m)' for t in resolved.teams])
    if resolved.phases:      matches.extend([f'phase:{p.project}/{p.phase_label}' for p in resolved.phases])
    if resolved.instruments: matches.extend([f'instr:{i.code}' for i in resolved.instruments])
    if resolved.datasets:    matches.extend([f'data:{d.code}' for d in resolved.datasets])
    icon = {'single': 'o', 'multi': '*', 'control': '-'}[tq['complexity']]
    status = ', '.join(matches) if matches else '(no matches)'
    print(f'  [{icon}] {tq["query"][:62]:<62s} -> {status}')

## 4. Select Models

Uncomment the models you want to benchmark. Results are saved to CSV so runs can be spread across sessions.

In [ ]:
models_to_test = {
    # Tier 1: Tiny (3-4B)
    'Phi-3 Mini':        ('ollama', 'phi3:mini'),
    'Phi-4 Mini':        ('ollama', 'phi4-mini'),
    'Qwen3-4B':          ('ollama', 'qwen3:4b'),
    # Tier 2: Small (7-9B)
    'Mistral-7B':        ('ollama', 'mistral'),
    'Llama3.1-8B':       ('ollama', 'llama3.1:8b'),
    'Qwen3-8B':          ('ollama', 'qwen3:8b'),
    'Gemma2-9B':         ('ollama', 'gemma2:9b'),
    # Tier 3: Medium (14B)
    'Qwen3-14B':         ('ollama', 'qwen3:14b'),
    'Phi-4':             ('ollama', 'phi4'),
    # Tier 4: Large (24-32B)
    'Mistral-Small-3.1': ('ollama', 'mistral-small3.1'),
    'Gemma2-27B':        ('ollama', 'gemma2:27b'),
    'Qwen3-32B':         ('ollama', 'qwen3:32b'),
    # Cloud
    # 'Claude Sonnet':   ('claude', 'claude-sonnet-4-6'),
}

MODEL_SIZE = {
    'Phi-3 Mini': 'tiny',  'Phi-4 Mini': 'tiny',   'Qwen3-4B': 'tiny',
    'Mistral-7B': 'small', 'Llama3.1-8B': 'small', 'Qwen3-8B': 'small', 'Gemma2-9B': 'small',
    'Qwen3-14B': 'medium', 'Phi-4': 'medium',
    'Mistral-Small-3.1': 'large', 'Gemma2-27B': 'large', 'Qwen3-32B': 'large',
    'Claude Sonnet': 'cloud',
}

total = len(models_to_test) * len(test_queries) * 2
print(f'Models: {list(models_to_test.keys())}')
print(f'Total LLM calls: {total}')

## 5. Run Benchmark

In [ ]:
async def run_one_mode(model_name, provider, model_id, queries, context_yaml_path=None):
    mode = 'context' if context_yaml_path else 'baseline'
    results = []
    for tq in queries:
        t0 = time.time()
        try:
            response = await run_arxiv_agent(
                tq['query'], llm_provider=provider, model_id=model_id,
                auto_pull=True, auto_unload=True, context_yaml_path=context_yaml_path,
            )
            latency_ms = (time.time() - t0) * 1000
            papers_df  = response.get('papers_df')
            cleaned_content, cleaned_author, cleaned_category = clean_components(
                response.get('query_content',  'NONE'),
                response.get('query_author',   'NONE'),
                response.get('query_category', 'NONE'),
            )
            vr    = validate_query_components(cleaned_content, cleaned_author, cleaned_category,
                                             len(papers_df) if papers_df is not None else 0)
            valid = vr['all_components_valid']
            reason = ''
            if not valid:
                for comp in ('content', 'author', 'category'):
                    if not vr[comp]['valid']:
                        reason = vr[comp].get('error', 'unknown')
                        break
            results.append({
                'Model': model_name, 'Mode': mode,
                'Input': tq['query'],
                'ContextTypes': ', '.join(tq['context_types']),
                'Complexity': tq['complexity'],
                'Content': cleaned_content, 'Author': cleaned_author, 'Category': cleaned_category,
                'Query': response.get('arxiv_query', ''),
                'Valid': valid, 'Papers': len(papers_df) if papers_df is not None else 0,
                'Reason': reason, 'Latency_ms': latency_ms,
            })
        except Exception as e:
            results.append({
                'Model': model_name, 'Mode': mode,
                'Input': tq['query'],
                'ContextTypes': ', '.join(tq['context_types']),
                'Complexity': tq['complexity'],
                'Content': 'ERROR', 'Author': 'ERROR', 'Category': 'ERROR', 'Query': 'ERROR',
                'Valid': False, 'Papers': 0,
                'Reason': str(e), 'Latency_ms': (time.time() - t0) * 1000,
            })
    return results


results_df = pd.DataFrame()

for model_name, (provider, model_id) in models_to_test.items():
    size = MODEL_SIZE.get(model_name, '?')
    print(f'\n{'='*72}\n  {model_name} ({size})\n{'='*72}')

    print('  Baseline ...')
    bl_rows = await run_one_mode(model_name, provider, model_id, test_queries)
    bl_df   = pd.DataFrame(bl_rows)
    bl_df['Size'] = size

    print('  Context-enriched ...')
    ct_rows = await run_one_mode(model_name, provider, model_id, test_queries, CONTEXT_YAML_PATH)
    ct_df   = pd.DataFrame(ct_rows)
    ct_df['Size'] = size

    results_df = pd.concat([results_df, bl_df, ct_df], ignore_index=True)
    print(f'  Baseline {bl_df["Valid"].sum()}/{len(bl_df)} valid   '
          f'Context {ct_df["Valid"].sum()}/{len(ct_df)} valid')

In [ ]:
if OVERWRITE_CSV or not os.path.exists(OUTPUT_CSV):
    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f'Saved {len(results_df)} rows -> {OUTPUT_CSV}')
else:
    existing = pd.read_csv(OUTPUT_CSV)
    combined = pd.concat([existing, results_df], ignore_index=True)
    combined.to_csv(OUTPUT_CSV, index=False)
    print(f'Appended {len(results_df)} rows -> {OUTPUT_CSV} (total: {len(combined)})')

# To reload from CSV without re-running the benchmark:
# results_df = pd.read_csv(OUTPUT_CSV)

---

# Analysis

## 6. Semantic Scoring Setup

The **structural validator** checks syntax — are `cat:` codes real? Are `au:` formats correct?
But this misses the point:

| Query | Baseline output | Context output |
|---|---|---|
| "Helios team papers" | `au:helios_t` — valid syntax, 0 useful papers | `au:Vasquez-Torres OR au:Nakamura OR ...` — correct |
| "During Phase 1 of Helios" | No date filter — valid syntax, wrong semantics | `submittedDate:[20200601 TO 20220331]` — correct |

Baseline can score **higher** structural validity because it outputs `NONE` or hallucinated names
(syntactically fine, semantically empty). Context tries to do the right thing and sometimes fails
the regex (apostrophe in O'Brien-Walsh, for example).

The following cells define rule-based semantic checks and compute a semantic score
that measures what actually matters.

In [ ]:
# ── Known ground truth ────────────────────────────────────────────────────────
HELIOS_MEMBERS    = ['Vasquez-Torres', 'Nakamura', 'Sharma-Patel', 'Brien-Walsh']
QUANTUMBIO_MEMBERS = ['Chen-Rodriguez', 'Okafor', 'Petrov']

HELIOS_P1 = ('20200601', '20220331')
HELIOS_P2 = ('20220401', '20240930')
QB_PILOT  = ('20230101', '20231231')

EXPECTED_CATEGORIES = {
    'SPARC':       ['astro-ph.GA', 'astro-ph.SR'],
    'CryoNet':     ['q-bio.BM', 'cs.LG', 'cs.CV'],
    'TurboPEC':    ['astro-ph.GA', 'astro-ph.HE'],
    'FoldX':       ['quant-ph', 'q-bio.BM', 'physics.chem-ph'],
    'APEX':        ['astro-ph.EP', 'astro-ph.IM'],
    'NEXUS-II':    ['astro-ph.HE', 'astro-ph.IM', 'astro-ph.CO'],
    'PolariScope': ['astro-ph.GA', 'astro-ph.SR', 'astro-ph.IM'],
    'PHOTON-Z':    ['astro-ph.CO', 'astro-ph.GA'],
    'SynBio-DB':   ['q-bio.BM', 'q-bio.GN'],
}


# ── Check functions ───────────────────────────────────────────────────────────
def check_team_expansion(author_str, members):
    if pd.isna(author_str) or str(author_str).upper() == 'NONE':
        return 0, len(members), []
    s = str(author_str)
    found = [m for m in members if re.search(m.replace("'", "'?").replace('-', '-?'), s, re.IGNORECASE)]
    return len(found), len(members), found


def check_or_join(author_str):
    s = str(author_str)
    return ' OR ' in s.upper() and ' AND ' not in s.upper()


def check_date_range(content_str, start, end):
    s = str(content_str) if not pd.isna(content_str) else ''
    return 'submittedDate:' in s, start in s, end in s


def check_categories(cat_str, expected):
    if pd.isna(cat_str) or str(cat_str).upper() == 'NONE':
        return 0, len(expected), []
    s = str(cat_str)
    matched = [c for c in expected if c in s]
    all_cats = re.findall(r'cat:([-\w.]+)', s)
    return len(matched), len(expected), all_cats


def check_no_author_leakage(content_str):
    if pd.isna(content_str) or str(content_str).upper() == 'NONE':
        return True
    s = str(content_str).lower()
    if 'au:' in s:
        return False
    for name in HELIOS_MEMBERS + QUANTUMBIO_MEMBERS:
        if name.lower() in s:
            return False
    return True


print('Scoring functions ready.')

In [ ]:
scores = []

for _, row in results_df.iterrows():
    ctx   = str(row.get('ContextTypes', ''))
    content  = str(row.get('Content',  'NONE'))
    author   = str(row.get('Author',   'NONE'))
    category = str(row.get('Category', 'NONE'))
    q_text   = str(row.get('Input', ''))

    s = {
        'Model': row['Model'], 'Size': row['Size'], 'Mode': row['Mode'],
        'Input': q_text, 'ContextTypes': ctx, 'Complexity': row['Complexity'],
        'Valid_structural': row['Valid'], 'Papers': row['Papers'],
    }

    # Team expansion
    if 'team' in ctx and 'control' not in ctx:
        members = HELIOS_MEMBERS if 'Helios' in q_text else QUANTUMBIO_MEMBERS
        n_found, n_exp, _ = check_team_expansion(author, members)
        s['team_members_found']    = n_found
        s['team_members_expected'] = n_exp
        s['team_expansion_rate']   = n_found / n_exp
        s['team_or_join']          = check_or_join(author)
        s['team_no_leakage']       = check_no_author_leakage(content)

    # Date resolution
    if 'project_phase' in ctx:
        if 'Phase 1 of the Helios' in q_text:
            has_d, has_s, has_e = check_date_range(content, *HELIOS_P1)
        elif 'Phase 2 of the Helios' in q_text:
            has_d, has_s, has_e = check_date_range(content, *HELIOS_P2)
        elif 'QuantumBio Pilot' in q_text:
            has_d, has_s, has_e = check_date_range(content, *QB_PILOT)
        else:
            has_d, has_s, has_e = False, False, False
        s['date_has_range']    = has_d
        s['date_correct_start'] = has_s
        s['date_correct_end']   = has_e
        s['date_fully_correct'] = has_s and has_e

    # Category correctness
    for term, expected in EXPECTED_CATEGORIES.items():
        if term.lower() in q_text.lower():
            n_match, n_exp, all_cats = check_categories(category, expected)
            s['cat_matched']    = n_match
            s['cat_expected']   = n_exp
            s['cat_match_rate'] = n_match / n_exp if n_exp else 0
            break

    scores.append(s)

scores_df = pd.DataFrame(scores)
print(f'Scored {len(scores_df)} rows')
semantic_cols = [c for c in scores_df.columns if c.startswith(('team_', 'date_', 'cat_'))]
print(f'Semantic columns: {semantic_cols}')

---

## 7. Team Expansion — The Primary Signal

This is the most diagnostic test for context injection. The Helios and QuantumBio teams are
**entirely fictional** — baseline mode cannot expand them. If context mode correctly injects
member names, the author component should list all members joined with OR.

A structurally valid `au:helios_t` in baseline mode finds zero papers and is semantically useless.
A structurally invalid `au:OBrien-Walsh OR au:Nakamura` actually works on arXiv.

In [ ]:
team_scores = scores_df[scores_df['team_members_expected'].notna()].copy()

print('=' * 72)
print('  TEAM EXPANSION — Baseline vs Context')
print('=' * 72)

for mode in ['baseline', 'context']:
    sub = team_scores[team_scores['Mode'] == mode]
    n             = len(sub)
    avg_rate      = sub['team_expansion_rate'].mean() * 100
    full_exp      = (sub['team_expansion_rate'] == 1.0).sum()
    any_exp       = (sub['team_expansion_rate'] > 0).sum()
    or_join       = sub['team_or_join'].sum()
    no_leak       = sub['team_no_leakage'].sum()
    struct_valid  = sub['Valid_structural'].sum()

    print(f'\n  {mode.upper()} ({n} team queries):')
    print(f'    Avg member coverage:      {avg_rate:5.1f}%')
    print(f'    Full expansion (all):     {full_exp}/{n} ({full_exp/n*100:.0f}%)')
    print(f'    Any expansion (>=1):      {any_exp}/{n} ({any_exp/n*100:.0f}%)')
    print(f'    Correct OR-join:          {or_join}/{n} ({or_join/n*100:.0f}%)')
    print(f'    No author->content leak:  {no_leak}/{n} ({no_leak/n*100:.0f}%)')
    print(f'    Structural validity:      {struct_valid}/{n} ({struct_valid/n*100:.0f}%)')

print('\n  Takeaway: baseline has HIGHER structural validity but ZERO team expansion.')
print('  Context produces semantically meaningful queries despite lower structural validity.')

In [ ]:
print(f'  {"Model":<22s} {"Members%":>9s} {"OR-join%":>9s} {"Struct%":>9s}')
print(f'  {"-"*22} {"-"*9} {"-"*9} {"-"*9}')

ctx_team = team_scores[team_scores['Mode'] == 'context']
for model in sorted(ctx_team['Model'].unique()):
    m   = ctx_team[ctx_team['Model'] == model]
    avg = m['team_expansion_rate'].mean() * 100
    orj = m['team_or_join'].mean() * 100
    stv = m['Valid_structural'].mean() * 100
    print(f'  {model:<22s} {avg:8.0f}% {orj:8.0f}% {stv:8.0f}%')

## 8. Date Resolution

Baseline *cannot* know that "Phase 1 of the Helios Project" = June 2020 – March 2022.
Context mode injects the exact date arithmetic. Did models follow it?

In [ ]:
date_scores = scores_df[scores_df['date_has_range'].notna()].copy()

print('=' * 72)
print('  DATE RESOLUTION — Baseline vs Context')
print('=' * 72)

for mode in ['baseline', 'context']:
    sub      = date_scores[date_scores['Mode'] == mode]
    n        = len(sub)
    has_any  = sub['date_has_range'].sum()
    fully_ok = sub['date_fully_correct'].sum()
    print(f'\n  {mode.upper()} ({n} phase queries):')
    print(f'    Any date range:     {has_any}/{n} ({has_any/n*100:.0f}%)')
    print(f'    Fully correct:      {fully_ok}/{n} ({fully_ok/n*100:.0f}%)')

print('\n  Per-model (context mode):')
print(f'  {"Model":<22s} {"Has date":>10s} {"Correct":>10s}')
print(f'  {"-"*22} {"-"*10} {"-"*10}')
ctx_date = date_scores[date_scores['Mode'] == 'context']
for model in sorted(ctx_date['Model'].unique()):
    m  = ctx_date[ctx_date['Model'] == model]
    hd = m['date_has_range'].mean() * 100
    fc = m['date_fully_correct'].mean() * 100
    print(f'  {model:<22s} {hd:9.0f}% {fc:9.0f}%')

## 9. Category Correctness

Did context injection steer models toward the correct arXiv categories for glossary terms,
instruments, and datasets?

In [ ]:
cat_scores = scores_df[scores_df['cat_expected'].notna()].copy()

print('=' * 72)
print('  CATEGORY CORRECTNESS — Baseline vs Context')
print('=' * 72)

for mode in ['baseline', 'context']:
    sub       = cat_scores[cat_scores['Mode'] == mode]
    n         = len(sub)
    avg_rate  = sub['cat_match_rate'].mean() * 100
    any_match = (sub['cat_matched'] > 0).sum()
    print(f'\n  {mode.upper()} ({n} queries with expected categories):')
    print(f'    Avg match rate:   {avg_rate:5.1f}%')
    print(f'    Any correct cat:  {any_match}/{n} ({any_match/n*100:.0f}%)')

print()
print(f'  {"Query":<55s} {"BL":>6s} {"CTX":>6s} {"Delta":>7s}')
print(f'  {"-"*55} {"-"*6} {"-"*6} {"-"*7}')

for q in cat_scores['Input'].unique():
    bl = cat_scores[(cat_scores['Input'] == q) & (cat_scores['Mode'] == 'baseline')]
    ct = cat_scores[(cat_scores['Input'] == q) & (cat_scores['Mode'] == 'context')]
    if bl.empty or ct.empty:
        continue
    bl_r  = bl['cat_match_rate'].mean() * 100
    ct_r  = ct['cat_match_rate'].mean() * 100
    delta = ct_r - bl_r
    print(f'  {q[:55]:<55s} {bl_r:5.0f}% {ct_r:5.0f}% {delta:+6.0f}%')

## 10. The Structural Paradox — Why Baseline Looks Better

The structural validator uses the regex `^au:[A-Za-z][\w-]*` which rejects apostrophes.
The Helios team includes `Marcus O'Brien-Walsh`. Context mode correctly expands the team
but the apostrophe fails the regex — **penalising models for doing the right thing**.

This explains why context mode can show lower structural validity than baseline, even when
it produces semantically correct and useful queries. The prompt fix (rule 4: drop apostrophes
from Irish/Celtic surnames) addresses this directly.

In [ ]:
print('=' * 72)
print("  THE O'BRIEN-WALSH APOSTROPHE PROBLEM")
print('=' * 72)

ctx = results_df[results_df['Mode'] == 'context']
ctx_brien = ctx[ctx['Author'].str.contains('Brien', na=False)]

def agg(g):
    with_apo    = g['Author'].str.contains("'", na=False)
    return pd.Series({
        'total':             len(g),
        'with_apostrophe':   with_apo.sum(),
        'without_apostrophe': (~with_apo).sum(),
        'valid_with_apo':    g[with_apo]['Valid'].sum(),
        'valid_without':     g[~with_apo]['Valid'].sum(),
    })

apo_counts = ctx_brien.groupby('Model').apply(agg).reset_index()

print("\n  Models that DROP the apostrophe (OBrien-Walsh) -> pass validation:")
for _, r in apo_counts[apo_counts['without_apostrophe'] > 0].iterrows():
    print(f"    {r['Model']:<22s}: {int(r['without_apostrophe'])}/{int(r['total'])} clean, {int(r['valid_without'])} pass")

print("\n  Models that KEEP the apostrophe (O'Brien-Walsh) -> fail validation:")
for _, r in apo_counts[apo_counts['with_apostrophe'] > 0].iterrows():
    print(f"    {r['Model']:<22s}: {int(r['with_apostrophe'])}/{int(r['total'])} with ', {int(r['valid_with_apo'])} pass")

# Quantify the impact on overall structural validity
total_ctx = len(ctx)
apo_failures = ctx_brien[ctx_brien['Author'].str.contains("'", na=False) & ~ctx['Valid']]
print(f'\n  Apostrophe failures account for {len(apo_failures)}/{total_ctx} context-mode failures')
print("  These are validator false negatives — model output is semantically correct.")

## 11. Composite Semantic Score

A 0–10 score that captures what structural validity misses:

| Dimension | Points | Condition |
|---|---|---|
| Content relevance | 0–2 | Has valid `all:` or `ti:` term |
| Team/author | 0–2 | Member coverage ≥ 75% with OR-join |
| Date resolution | 0–2 | Correct `submittedDate` range |
| Category correctness | 0–2 | ≥ 50% of expected categories |
| Clean separation | 0–2 | No author names leaking into content |

Dimensions that don't apply to a query get full credit automatically.

In [ ]:
def compute_semantic_score(row):
    score, max_score, breakdown = 0, 0, {}

    # 1. Content relevance
    max_score += 2
    content = str(row.get('Content', 'NONE'))
    if content.upper() != 'NONE' and any(op in content for op in ['all:', 'ti:', 'id:']):
        breakdown['content'] = 2; score += 2
    elif content.upper() == 'NONE' and 'control_author' in str(row.get('ContextTypes', '')):
        breakdown['content'] = 1; score += 1
    else:
        breakdown['content'] = 0

    # 2. Team/author
    ctx = str(row.get('ContextTypes', ''))
    if 'team' in ctx and 'control' not in ctx:
        max_score += 2
        rate = row.get('team_expansion_rate', 0)
        or_ok = row.get('team_or_join', False)
        pts = 0
        if pd.notna(rate) and rate >= 0.5:  pts = 1
        if pd.notna(rate) and rate >= 0.75 and or_ok: pts = 2
        breakdown['team'] = pts; score += pts

    # 3. Date resolution
    if 'project_phase' in ctx:
        max_score += 2
        fully = row.get('date_fully_correct', False)
        has_d = row.get('date_has_range', False)
        pts = 2 if fully else (1 if has_d else 0)
        breakdown['date'] = pts; score += pts

    # 4. Category correctness
    cat_rate = row.get('cat_match_rate', None)
    if pd.notna(cat_rate):
        max_score += 2
        if cat_rate >= 0.5:            pts = 2
        elif cat_rate > 0:             pts = 1
        elif str(row.get('Category', 'NONE')).upper() == 'NONE': pts = 1
        else:                          pts = 0
        breakdown['category'] = pts; score += pts

    # 5. Clean separation
    max_score += 2
    no_leak = row.get('team_no_leakage', None)
    pts = 2 if (pd.isna(no_leak) or no_leak) else 0
    breakdown['clean'] = pts; score += pts

    return round((score / max_score * 10) if max_score else 0, 1)


scores_df['semantic_score'] = scores_df.apply(compute_semantic_score, axis=1)

print('=' * 72)
print('  COMPOSITE SEMANTIC SCORE (0-10)')
print('=' * 72)

for mode in ['baseline', 'context']:
    sub = scores_df[scores_df['Mode'] == mode]
    print(f'  {mode.upper()}: mean={sub["semantic_score"].mean():.1f}/10, '
          f'median={sub["semantic_score"].median():.1f}/10')

ctx_dep = scores_df[~scores_df['ContextTypes'].str.contains('control')]
print('\n  Context-dependent queries only:')
for mode in ['baseline', 'context']:
    sub = ctx_dep[ctx_dep['Mode'] == mode]
    print(f'    {mode.upper()}: mean={sub["semantic_score"].mean():.1f}/10')

In [ ]:
print('=' * 90)
print('  STRUCTURAL vs SEMANTIC — The Full Picture (context-dependent queries)')
print('=' * 90)

ctx_dep = scores_df[~scores_df['ContextTypes'].str.contains('control')]
rows = []

for model in sorted(ctx_dep['Model'].unique()):
    size = ctx_dep[ctx_dep['Model'] == model]['Size'].iloc[0]
    bl   = ctx_dep[(ctx_dep['Model'] == model) & (ctx_dep['Mode'] == 'baseline')]
    ct   = ctx_dep[(ctx_dep['Model'] == model) & (ctx_dep['Mode'] == 'context')]

    ct_team = ct[ct['team_expansion_rate'].notna()]
    ct_date = ct[ct['date_has_range'].notna()]
    team_pct = ct_team['team_expansion_rate'].mean() * 100 if len(ct_team) else np.nan
    date_pct = ct_date['date_fully_correct'].mean() * 100 if len(ct_date) else np.nan

    rows.append({
        'Model':       model,
        'Size':        size,
        'BL Struct%':  f'{bl["Valid_structural"].mean()*100:.0f}',
        'CTX Struct%': f'{ct["Valid_structural"].mean()*100:.0f}',
        'BL Sem':      f'{bl["semantic_score"].mean():.1f}',
        'CTX Sem':     f'{ct["semantic_score"].mean():.1f}',
        'Delta Sem':   f'{ct["semantic_score"].mean()-bl["semantic_score"].mean():+.1f}',
        'Team%':       f'{team_pct:.0f}' if not np.isnan(team_pct) else '-',
        'Date%':       f'{date_pct:.0f}' if not np.isnan(date_pct) else '-',
    })

table = pd.DataFrame(rows)
print(table.to_string(index=False))

## 12. Does Context Complexity Break Models?

Multi-context queries ask models to handle team expansion + date ranges + glossary
simultaneously. As instructions grow, smaller models may fail to follow all of them.
This section checks whether combining multiple context types degrades quality.

In [ ]:
print('=' * 72)
print('  VALIDITY BY QUERY COMPLEXITY')
print('=' * 72)

for complexity in ['single', 'multi', 'control']:
    sub = results_df[results_df['Complexity'] == complexity]
    if sub.empty:
        continue
    n_queries = sub['Input'].nunique()
    print(f'\n  {complexity.upper()} ({n_queries} queries):')
    for mode in ['baseline', 'context']:
        m   = sub[sub['Mode'] == mode]
        n   = len(m)
        v   = m['Valid'].sum()
        pct = v / n * 100 if n else 0
        print(f'    {mode:>10s}: {v}/{n} valid ({pct:5.1f}%)')

# Semantic score by complexity
print('\n  Semantic score by complexity (context mode):')
for complexity in ['single', 'multi']:
    sub = scores_df[(scores_df['Complexity'] == complexity) & (scores_df['Mode'] == 'context')]
    sub = sub[~sub['ContextTypes'].str.contains('control')]
    if not sub.empty:
        print(f'    {complexity.upper()}: mean semantic score = {sub["semantic_score"].mean():.1f}/10')

In [ ]:
print('=' * 72)
print('  SEMANTIC SCORE BY CONTEXT TYPE (context mode only)')
print('=' * 72)

exploded = scores_df[scores_df['Mode'] == 'context'].copy()
exploded['ContextTypes'] = exploded['ContextTypes'].str.split(', ')
exploded = exploded.explode('ContextTypes')

type_order = ['glossary', 'team', 'project_phase', 'instrument', 'dataset']
print(f'  {"Type":<16s} {"Struct Valid%":>14s} {"Semantic/10":>12s}')
print(f'  {"-"*16} {"-"*14} {"-"*12}')

for ct in type_order:
    sub = exploded[exploded['ContextTypes'] == ct]
    if sub.empty:
        continue
    sv  = sub['Valid_structural'].mean() * 100
    sem = sub['semantic_score'].mean()
    print(f'  {ct:<16s} {sv:13.1f}% {sem:11.1f}')

## 13. Model Size vs Context Benefit

Do larger models follow contextual instructions more reliably?
We compare semantic score improvement (context – baseline) across size tiers.

In [ ]:
print('=' * 72)
print('  CONTEXT BENEFIT BY MODEL SIZE (context-dependent queries)')
print('=' * 72)

ctx_dep = scores_df[~scores_df['ContextTypes'].str.contains('control')]

for size in ['tiny', 'small', 'medium', 'large', 'cloud']:
    sub = ctx_dep[ctx_dep['Size'] == size]
    if sub.empty:
        continue
    bl   = sub[sub['Mode'] == 'baseline']
    ct   = sub[sub['Mode'] == 'context']
    delta_sem    = ct['semantic_score'].mean() - bl['semantic_score'].mean()
    delta_struct = (ct['Valid_structural'].mean() - bl['Valid_structural'].mean()) * 100
    models = ', '.join(sorted(sub['Model'].unique()))
    print(f'\n  {size.upper()} ({models}):')
    print(f'    BL semantic:  {bl["semantic_score"].mean():.1f}/10   '
          f'CTX semantic: {ct["semantic_score"].mean():.1f}/10   '
          f'Delta: {delta_sem:+.1f}')
    print(f'    BL struct%:   {bl["Valid_structural"].mean()*100:.1f}%     '
          f'CTX struct%:  {ct["Valid_structural"].mean()*100:.1f}%     '
          f'Delta: {delta_struct:+.1f}pp')

## 14. Failure Analysis — Context Mode

What breaks when context is injected? Categorise failures by reason.

In [ ]:
ctx_failed = results_df[(results_df['Mode'] == 'context') & (~results_df['Valid'])]

print('=' * 72)
print('  FAILURE BREAKDOWN — context mode only')
print('=' * 72)

if ctx_failed.empty:
    print('  All context-enriched tests passed!')
else:
    reason_counts = ctx_failed.groupby(['Model', 'Reason']).size().reset_index(name='Count')
    print('\n  Failures by model and reason:')
    print(reason_counts.to_string(index=False))

    print(f'\n  Sample failures ({min(8, len(ctx_failed))} shown):')
    for _, r in ctx_failed.head(8).iterrows():
        print(f"\n    [{r['Model']}] {r['Reason']}")
        print(f"      Query:    {r['Input'][:65]}")
        print(f"      Author:   {str(r['Author'])[:65]}")
        print(f"      Category: {str(r['Category'])[:65]}")

---

## 15. Article-Ready Tables

In [ ]:
ctx_dep = scores_df[~scores_df['ContextTypes'].str.contains('control')]

summary_rows = []
for model in sorted(ctx_dep['Model'].unique()):
    size = ctx_dep[ctx_dep['Model'] == model]['Size'].iloc[0]
    bl   = ctx_dep[(ctx_dep['Model'] == model) & (ctx_dep['Mode'] == 'baseline')]
    ct   = ctx_dep[(ctx_dep['Model'] == model) & (ctx_dep['Mode'] == 'context')]

    ct_team = ct[ct['team_expansion_rate'].notna()]
    ct_date = ct[ct['date_has_range'].notna()]
    team_rate = ct_team['team_expansion_rate'].mean() * 100 if len(ct_team) else np.nan
    date_rate = ct_date['date_fully_correct'].mean() * 100  if len(ct_date) else np.nan

    summary_rows.append({
        'Model':        model,
        'Size':         size,
        'BL Struct%':   f'{bl["Valid_structural"].mean()*100:.0f}%',
        'CTX Struct%':  f'{ct["Valid_structural"].mean()*100:.0f}%',
        'BL Sem/10':    f'{bl["semantic_score"].mean():.1f}',
        'CTX Sem/10':   f'{ct["semantic_score"].mean():.1f}',
        'Sem Delta':    f'{ct["semantic_score"].mean()-bl["semantic_score"].mean():+.1f}',
        'Team Exp%':    f'{team_rate:.0f}%' if not np.isnan(team_rate) else '-',
        'Date Res%':    f'{date_rate:.0f}%' if not np.isnan(date_rate) else '-',
        'BL Papers':    int(results_df[(results_df['Model']==model)&(results_df['Mode']=='baseline')]['Papers'].sum()),
        'CTX Papers':   int(results_df[(results_df['Model']==model)&(results_df['Mode']=='context')]['Papers'].sum()),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

In [ ]:
print('Context injection — structural validity (solid) vs semantic score (hollow)')
print()

ctx_dep = scores_df[~scores_df['ContextTypes'].str.contains('control')]

for model in sorted(ctx_dep['Model'].unique()):
    bl = ctx_dep[(ctx_dep['Model'] == model) & (ctx_dep['Mode'] == 'baseline')]
    ct = ctx_dep[(ctx_dep['Model'] == model) & (ctx_dep['Mode'] == 'context')]

    bl_struct = bl['Valid_structural'].mean() * 100
    ct_struct = ct['Valid_structural'].mean() * 100
    bl_sem    = bl['semantic_score'].mean() * 10   # scale to 0-100
    ct_sem    = ct['semantic_score'].mean() * 10

    def bar(v, c): return c * int(v / 2.5)

    print(f'  {model}:')
    print(f'    Struct  BL: {bar(bl_struct, chr(9608)):<42s} {bl_struct:5.1f}%')
    print(f'    Struct CTX: {bar(ct_struct, chr(9617)):<42s} {ct_struct:5.1f}%')
    print(f'    Semant  BL: {bar(bl_sem,    chr(9608)):<42s} {bl_sem/10:5.1f}/10')
    print(f'    Semant CTX: {bar(ct_sem,    chr(9617)):<42s} {ct_sem/10:5.1f}/10')
    print()

## 16. Article Examples — Generated Query Showcase

In [ ]:
showcase_queries = [
    'Papers about SPARC analysis methods for galaxy surveys',
    'Papers published by the Helios team about dark matter in galaxy clusters',
    'Papers published during Phase 1 of the Helios Project',
    'Papers by the Helios team about SPARC analysis using NEXUS-II data',
]
showcase_models = list(models_to_test.keys())[:4]

for q_text in showcase_queries:
    tq = next((t for t in test_queries if t['query'] == q_text), None)
    if not tq:
        continue
    print(f'\n{"="*90}')
    print(f'  USER: "{q_text}"')
    print(f'  Context types: {", ".join(tq["context_types"])}')
    print(f'{"="*90}')

    for model_name in showcase_models:
        for mode in ['baseline', 'context']:
            row = results_df[
                (results_df['Model'] == model_name) &
                (results_df['Mode']  == mode) &
                (results_df['Input'] == q_text)
            ]
            if row.empty:
                continue
            r = row.iloc[0]
            status = 'V' if r['Valid'] else 'X'
            print(f'\n  {model_name} ({mode}): [{status}]')
            print(f'    Content:  {str(r["Content"])[:80]}')
            print(f'    Author:   {str(r["Author"])[:80]}')
            print(f'    Category: {str(r["Category"])[:80]}')
            print(f'    Combined: {str(r["Query"])[:100]}')

---

## 17. LLM-as-Judge

For nuanced cases where rule-based scoring isn't enough. Uses a large local model
to evaluate semantic quality on a 0–10 scale.

Set `RUN_JUDGE = True` to run. Warning: this makes `len(results_df)` additional LLM calls.

In [ ]:
JUDGE_PROVIDER = 'ollama'
JUDGE_MODEL    = 'qwen3:32b'
RUN_JUDGE      = True
JUDGE_CSV      = 'article_judge_results.csv'

JUDGE_PROMPT = """
You are evaluating whether an arXiv search query will retrieve the papers the user asked for.
Your focus is on RETRIEVAL QUALITY — will running this query on the arXiv API return relevant papers?

USER REQUEST: {user_input}

{context_block}

GENERATED QUERY:
  Content:  {content}
  Author:   {author}
  Category: {category}
  Combined: {combined}

Score each dimension 0-2 based on how well the context was integrated and how useful the query will be:

1. CONTENT (0-2)
   Look at: the Content component and the full Combined query.
   Reference: the "Domain-specific context" and "Project phase dates" sections above (if any).
   2 = content correctly captures the topic AND uses any injected term expansions or date ranges
   1 = partially correct — topic captured but injected expansions ignored or misapplied
   0 = wrong topic, or date range was provided in context but not used in the query

2. AUTHORS (0-2)
   Look at: the Author component.
   Reference: the "Team definitions" section above (if any) — it lists the exact expected members.
   When a NAMED TEAM is mentioned and member list was provided in context:
     2 = all (or nearly all) members listed joined with OR  [e.g. au:Smith OR au:Jones OR au:Lee]
         OR is CORRECT — a paper needs only ONE team member as author to be relevant.
         NEVER penalise OR-joining. It is the right operator here.
     1 = some members listed with OR but noticeably incomplete
     0 = team not expanded at all (NONE, single hallucinated token, or AND-joined members)
   When a NAMED TEAM is mentioned but NO context was provided (baseline mode):
     0 = always. The LLM cannot know fictional team members. No partial credit.
         NONE is also 0 — team expansion was required and is simply absent.
   When no specific author is mentioned:
     2 = NONE (correctly omitted)
   When a real individual author is mentioned:
     2 = correct au: format, 1 = minor format issue, 0 = wrong or missing

3. DATES (0-2)
   Look at: the Content component (date ranges go in content as submittedDate:[...]).
   Reference: the "Project phase dates" section above — it gives the exact expected date range.
   2 = correct submittedDate:[YYYYMMDD TO YYYYMMDD] range matching the phase, or correctly
       absent when no phase was mentioned
   1 = date range present but boundaries don't match the injected dates
   0 = a phase was mentioned and date context was provided, but no submittedDate in the query

4. CATEGORIES (0-2)
   Look at: the Category component.
   Reference: the "Category hints" or "REQUIRED category hints" section above (if any).
   2 = at least one of the expected categories is present, or NONE when the query is broad
   1 = categories present but from the wrong subdomain
   0 = category context was provided but completely ignored, or wrong-domain codes used

5. RETRIEVAL IMPACT (0-2)
   Considering ALL of the above: will running this Combined query on the arXiv API return
   the papers the user actually asked for?
   2 = yes — the query correctly scopes authors, dates, topic, and categories
   1 = partially — finds some relevant papers but misses key constraints or adds noise
   0 = no — query is too generic, uses wrong authors, or misses the main constraint entirely

Set total = content + authors + dates + categories + retrieval_impact.

Respond with ONLY JSON (no markdown, no preamble):
{{"content": 0, "authors": 0, "dates": 0, "categories": 0, "retrieval_impact": 0, "total": 0, "reasoning": "one sentence on the main gap"}}
"""

print(f'Judge: {JUDGE_MODEL}   Run: {RUN_JUDGE}')

In [ ]:
results_df = pd.read_csv("context_benchmark_results.csv")

In [ ]:
results_df.sample(4)

In [ ]:
judge_df = None

if RUN_JUDGE:
    from src.application.factories import LLMProviderFactory
    judge    = LLMProviderFactory.create(JUDGE_PROVIDER, JUDGE_MODEL, auto_pull=True)
    jresults = []

    for i, (_, row) in enumerate(results_df.iterrows()):
        mode = row['Mode']


        if mode == 'context':
            resolved = resolver.resolve(row['Input'])
            print(resolved)
            if resolved.has_matches:
                content_ctx  = resolved.format_for_prompt('content').strip()
                author_ctx   = resolved.format_for_prompt('author').strip()
                category_ctx = resolved.format_for_prompt('category').strip()

                sections = ['--- CONTEXT PROVIDED TO THE MODEL ---']
                if content_ctx:
                    sections.append('[Content prompt received:]\n' + content_ctx)
                if author_ctx:
                    sections.append('[Author prompt received:]\n' + author_ctx)
                if category_ctx:
                    sections.append('[Category prompt received:]\n' + category_ctx)
                sections.append('--- END OF CONTEXT ---')
                ctx_block = '\n\n'.join(sections)
            else:
                ctx_block = (
                    '--- CONTEXT MODE: no matching entries for this query ---\n'
                    'No external context was injected. Evaluate as if baseline.'
                )
        else:
            ctx_block = (
                '--- BASELINE MODE: no external context was provided ---\n'
                'The model had to interpret all terms from training data alone.\n'
                'Fictional team names (Helios, QuantumBio), project phase dates, and\n'
                'custom instruments/datasets are unknown to the model.\n'
                'Any team expansion, date range, or instrument category in the output\n'
                'is a hallucination — there is no partial credit for guessing.'
            )

        prompt = JUDGE_PROMPT.format(
            user_input=row['Input'], mode=mode, context_block=ctx_block,
            content=row.get('Content', 'NONE'), author=row.get('Author', 'NONE'),
            category=row.get('Category', 'NONE'), combined=row.get('Query', ''),
        )
        try:
            raw = judge.respond(prompt)['content'].strip()
            raw = raw.replace('```json', '').replace('```', '').strip()
            result = json.loads(raw)
            result['parse_error'] = False
        except Exception as e:
            result = {'content': -1, 'authors': -1, 'dates': -1, 'categories': -1,
                      'retrieval_impact': -1, 'total': -1,
                      'reasoning': str(e), 'parse_error': True}

        result.update({'Model': row['Model'], 'Mode': mode,
                       'Input': row['Input'], 'ContextTypes': row.get('ContextTypes', '')})
        jresults.append(result)

        if (i + 1) % 20 == 0:
            print(f'  [{i+1}/{len(results_df)}] {row["Model"]:<18s} {mode:<10s} total={result.get("total", "?")}')

    judge_df = pd.DataFrame(jresults)
    judge_df.to_csv(JUDGE_CSV, index=False)
    print(f'Saved {len(judge_df)} judgments -> {JUDGE_CSV}')

elif os.path.exists(JUDGE_CSV):
    judge_df = pd.read_csv(JUDGE_CSV)
    print(f'Loaded existing judgments from {JUDGE_CSV}: {len(judge_df)} rows')
else:
    print('Judge not running. Set RUN_JUDGE = True to enable.')

In [ ]:
if judge_df is not None:
    valid_j   = judge_df[~judge_df.get('parse_error', pd.Series([False]*len(judge_df))).fillna(False)]
    ctx_dep_j = valid_j[~valid_j['ContextTypes'].str.contains('control', na=False)]

    print('=' * 72)
    print('  LLM JUDGE SCORES — Retrieval Quality (context-dependent queries)')
    print('=' * 72)

    dims = ['content', 'authors', 'dates', 'categories', 'retrieval_impact', 'total']

    for mode in ['baseline', 'context']:
        sub = ctx_dep_j[ctx_dep_j['Mode'] == mode]
        if sub.empty:
            continue
        print(f'\n  {mode.upper()} ({len(sub)} rows):')
        for d in dims:
            if d in sub.columns:
                print(f'    {d:<18s}: {sub[d].mean():.2f}/2' if d != 'total'
                      else f'    {d:<18s}: {sub[d].mean():.1f}/10')

    print('\n  Per-model judge scores (context mode):')
    print(f'  {"Model":<22s} {"authors":>8s} {"dates":>8s} {"cats":>8s} {"impact":>8s} {"total":>8s}')
    print(f'  {"-"*22} {"-"*8} {"-"*8} {"-"*8} {"-"*8} {"-"*8}')
    for model in sorted(ctx_dep_j['Model'].unique()):
        m = ctx_dep_j[(ctx_dep_j['Model'] == model) & (ctx_dep_j['Mode'] == 'context')]
        def avg(col): return f'{m[col].mean():.2f}' if col in m.columns else '-'
        print(f'  {model:<22s} {avg("authors"):>8s} {avg("dates"):>8s} '
              f'{avg("categories"):>8s} {avg("retrieval_impact"):>8s} {avg("total"):>8s}')

    # Baseline vs context delta on retrieval_impact
    if 'retrieval_impact' in ctx_dep_j.columns:
        print('\n  Retrieval impact delta (context - baseline) per model:')
        for model in sorted(ctx_dep_j['Model'].unique()):
            bl = ctx_dep_j[(ctx_dep_j['Model'] == model) & (ctx_dep_j['Mode'] == 'baseline')]
            ct = ctx_dep_j[(ctx_dep_j['Model'] == model) & (ctx_dep_j['Mode'] == 'context')]
            if bl.empty or ct.empty:
                continue
            delta = ct['retrieval_impact'].mean() - bl['retrieval_impact'].mean()
            print(f'    {model:<22s}: {delta:+.2f}')
else:
    print('No judge results available.')